# Data Ingestion
This file performs the data ingestion pipeline to ingest both Web-based and PDF resources into a document

### Step 1: Setting up the environment
This section sets up the required imports, environment variables for the data ingestion pipeline.

In [41]:
import json
import logging
import os
import sys

from dotenv import load_dotenv
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

sys.path.append(os.path.abspath(".."))

from app.backend.prepdocslib.extract_docs import extract_docs
from app.backend.prepdocslib.split_docs import split_docs

In [42]:
# Configurable Settings
save_vector_store: bool = True

In [43]:
# Load Environment Variables
load_dotenv()

# Set up Embedding model
OPENAI_EMB_MODEL = os.getenv("OPENAI_EMB_MODEL", "text-embedding-3-small")

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s - %(message)s",
)

In [44]:
CWD = os.getcwd()
ROOT = os.path.abspath(os.path.join(CWD, ".."))


# Create Function to create a directory
def create_folder(relative_path: str):
    """
    Helper function to create a folder if it does not exists and return file path

    Args:
        relative_path (str): Relative Path with respect to root

    Returns:
        str: String containing absolute file path
    """
    path = os.path.join(ROOT, relative_path)
    try:
        os.makedirs(path, exist_ok=True)
        print(f"Ensured directory '{path}' exists")
    except OSError as exc:
        print(f"Exception faced when creating folder '{path}': {exc}")
    return os.path.abspath(path)


# Set up necessary saving points
FAISS_DIR = create_folder(relative_path="app/backend/database/faiss")

Ensured directory 'c:\DBTT\DBTT_G5T1\app/backend/database/faiss' exists


### Step 2: Extract and Clean Documents
Extracting all documents from approved URLs and PDFs. To simplify the notebook, the same function used in the backend application will be imported here as a module. 

In [45]:
# Running Extract Documents
# urls is set to True to extract URLs in the `approved_sources.json` within the `app/backend/prepdocslib/docs` directory
# pdfs is set to True to extract PDFs within the `app/backend/prepdocslib/docs` directory
docs, config = extract_docs(urls=True, pdfs=True)

INFO - Extracting all documents
INFO - Extracting documents from approved URLs
INFO - Successfully extracted from URLs
INFO - Extracting documents from approved PDFs
INFO - Successfully extracted from PDFs
INFO - Successfully extracted all documents


In [46]:
# View total number of Documents
print(f"Total number of documents extracted: {len(docs)}")

# View sample document
print(json.dumps(docs[0].model_dump(), indent=4))

Total number of documents extracted: 46
{
    "id": null,
    "metadata": {
        "source": "https://www.healthhub.sg/health-conditions/acne",
        "title": "Acne",
        "description": "Learn about symptoms and causes of acne, self help treatment options, medication and when to seek medical help.",
        "language": "en"
    },
    "page_content": "Acne\n\n \n\nClear Search\n\nSearch\n\nSign up now \n\nMy e-Services \n\nMy e-Services\n\nSearch\n\nMenu\n\nHealthcare Directory\n\nHelp Centre\n\n \n\n \n\n\r\n Highlights & Insights\r\n \n\nExpert Voices\n\nHealth Safety Advisory\n\n\r\n Latest\r\n \n\n\r\n Latest\r\n \n\nQ&A: Wrist and Chronic Back Pain\nRead more\n\nQ&A: Skin Bumps\nRead more\n\n\r\n Trending\r\n \n\n\r\n Trending\r\n \n\nQ&A: Spicy Food and Big Appetite\nRead more\n\nQ&A: Expired Health Supplements and Eggs in Diet\nRead more\n\n\r\n Health Conditions\r\n \n\nAcne\n\nCold Sores\n\nConstipation\n\nCardiovascular Disease\n\nCough and Cold\n\nDiarrhoea\n\nHeadach

In [47]:
# View Configuration (to be used when updating vector store)
print(json.dumps(config, indent=4))

{
    "urls": [
        "https://www.healthhub.sg/health-conditions/acne",
        "https://www.healthhub.sg/health-conditions/cold-sores",
        "https://www.healthhub.sg/health-conditions/constipation",
        "https://www.healthhub.sg/health-conditions/heart-arrhythmias",
        "https://www.healthhub.sg/health-conditions/heart-failure-medication",
        "https://www.healthhub.sg/health-conditions/coronary-artery-disease",
        "https://www.healthhub.sg/health-conditions/heart-failure-basic-dietary-guideline",
        "https://www.healthhub.sg/health-conditions/heart-failure-monitoring-fluid-and-salt-intake",
        "https://www.healthhub.sg/health-conditions/heart-failure-signs-symptoms-diagnosis",
        "https://www.healthhub.sg/health-conditions/heart-failure-having-regular-exercise",
        "https://www.healthhub.sg/health-conditions/heart-failure-monitoring-weight-and-blood-pressure",
        "https://www.healthhub.sg/health-conditions/heart-failure-alcohol-and-smo

### Step 3: Split Documents
Documents are recursively split with overlap into chunks. This ensures keeping of semantic context and ensures more effective retrieval.

In [48]:
# Define chunk size and chunk overlap
chunk_size = 1000
chunk_overlap = 200

# Split documents
chunks = split_docs(documents=docs, chunk_size=chunk_size, chunk_overlap=chunk_overlap)

In [49]:
# View total number of chunks
print(f"Total number of chunks after splitting: {len(chunks)}")

# View sample chunk
print(json.dumps(chunks[0].model_dump(), indent=4))

Total number of chunks after splitting: 553
{
    "id": null,
    "metadata": {
        "source": "https://www.healthhub.sg/health-conditions/acne",
        "title": "Acne",
        "description": "Learn about symptoms and causes of acne, self help treatment options, medication and when to seek medical help.",
        "language": "en"
    },
    "page_content": "Acne\n\n \n\nClear Search\n\nSearch\n\nSign up now \n\nMy e-Services \n\nMy e-Services\n\nSearch\n\nMenu\n\nHealthcare Directory\n\nHelp Centre\n\n \n\n \n\n\r\n Highlights & Insights\r\n \n\nExpert Voices\n\nHealth Safety Advisory\n\n\r\n Latest\r\n \n\n\r\n Latest\r\n \n\nQ&A: Wrist and Chronic Back Pain\nRead more\n\nQ&A: Skin Bumps\nRead more\n\n\r\n Trending\r\n \n\n\r\n Trending\r\n \n\nQ&A: Spicy Food and Big Appetite\nRead more\n\nQ&A: Expired Health Supplements and Eggs in Diet\nRead more\n\n\r\n Health Conditions\r\n \n\nAcne\n\nCold Sores\n\nConstipation\n\nCardiovascular Disease\n\nCough and Cold\n\nDiarrhoea\n\nHea

### Step 4: Embedding and create vector store
This section defines an embedding model, which we will use to create the vector store.

In [50]:
# Set up embeddings and vector store and return
embeddings = OpenAIEmbeddings(model=OPENAI_EMB_MODEL)
vectorstore = FAISS.from_documents(chunks, embeddings)

INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [51]:
if save_vector_store:
    # Set up local vector store
    path = os.path.join(FAISS_DIR, "faiss_index")
    vectorstore.save_local(path)

In [52]:
# Update metadata and save configuration
config["metadata"] = {
    "chunk_size": chunk_size,
    "chunk_overlap": chunk_overlap,
    "embedding_model": OPENAI_EMB_MODEL,
}
print(json.dumps(config, indent=4))
with open(os.path.join(FAISS_DIR, "faiss_config.json"), "w") as f:
    json.dump(config, f, indent=2)

{
    "urls": [
        "https://www.healthhub.sg/health-conditions/acne",
        "https://www.healthhub.sg/health-conditions/cold-sores",
        "https://www.healthhub.sg/health-conditions/constipation",
        "https://www.healthhub.sg/health-conditions/heart-arrhythmias",
        "https://www.healthhub.sg/health-conditions/heart-failure-medication",
        "https://www.healthhub.sg/health-conditions/coronary-artery-disease",
        "https://www.healthhub.sg/health-conditions/heart-failure-basic-dietary-guideline",
        "https://www.healthhub.sg/health-conditions/heart-failure-monitoring-fluid-and-salt-intake",
        "https://www.healthhub.sg/health-conditions/heart-failure-signs-symptoms-diagnosis",
        "https://www.healthhub.sg/health-conditions/heart-failure-having-regular-exercise",
        "https://www.healthhub.sg/health-conditions/heart-failure-monitoring-weight-and-blood-pressure",
        "https://www.healthhub.sg/health-conditions/heart-failure-alcohol-and-smo

### Step 5: Load Vector Store and invoke
Loading vector store from the save location and testing its similarity test.

In [53]:
# Load vector store in
vector_store_path = os.path.join(FAISS_DIR, "faiss_index")
vectorstore = FAISS.load_local(
    vector_store_path, embeddings, allow_dangerous_deserialization=True
)

In [54]:
# Specify loading parameters
user_query = "What is Heart Disease?"  # Specify query to search for in vectorstore
num_sources = 4  # Specifies number of sources to retrieve

# Extract sources
sources = vectorstore.similarity_search(user_query, k=num_sources)

INFO - HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


In [55]:
# View Sources
for source in sources:
    print(json.dumps(source.model_dump(), indent=4))

{
    "id": "1d9b7a54-6406-4953-a5e4-8c47a7f39ee4",
    "metadata": {
        "source": "https://www.healthhub.sg/health-conditions/heart-failure-multiple-chronic-conditions-patients",
        "title": "Heart Failure: Multiple Chronic Conditions",
        "description": "A study on heart failure patients in Asia has identified patterns of chronic conditions like hypertension and coronary artery disease.",
        "language": "en"
    },
    "page_content": "Heart failure is a condition in which the heart loses the ability to pump enough blood to the body\u2019s tissues.Multimorbidity and ComorbidityMultimorbidity, also known as multiple comorbidities (more than one illness or disease occurring in one person simultaneously) or multiple chronic conditions, is common in patients with heart failure especially in Asia, where almost two-thirds of patients with heart failure were found to have multimorbidity.Multimorbidity can impede survival and complicate the diagnosis, treatment and outcom